# ACLED Data Exploration & Cleaning

This notebook runs **Steps 1 and 2** of the Myanmar territorial control pipeline.

| Step | Source script | What it does | Key output |
|------|--------------|--------------|------------|
| 1 | `01_explore_clean.py` | Load, profile, and clean 4-country ACLED CSVs | `data/processed/acled_clean.parquet` |
| 2 | `02_first_actor_analysis.py` | Temporal and spatial profile of the top first actor per country | `output/01_first_actor_analysis.md`, timeline & map PNGs |

Run cells top to bottom. Step 2 reads the parquet written by Step 1.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import NamedTuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
from scipy import stats

try:
    import contextily as ctx
    HAS_CONTEXTILY = True
except ImportError:
    HAS_CONTEXTILY = False

np.random.seed(20260428)

In [ ]:
ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = ROOT / "output"
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Data Inventory, Profiling, and Cleaning

**Input:** `data/raw/acled_*.csv` — four files covering Ecuador, Myanmar, Nigeria, and Somalia, all downloaded from the ACLED API on 2026-04-22.

**Output:**
- `data/processed/acled_clean.parquet` — combined cleaned dataset, read by all downstream notebooks.
- `data/processed/00_data_profile.md` — full column profile, geo-precision tables, and cleaning log.

Cleaning applies three steps per country: parse `event_date` (dropping any unparseable rows), deduplicate on `event_id_cnty`, and strip whitespace from `country`. No rows are dropped for missingness — geo-precision filtering is deferred to Step 2 so the profile captures the full raw picture first.

In [ ]:
GEO1_FLOOR = 50.0

STANDARD_COLS = {
    "event_id_cnty", "event_date", "year", "event_type", "sub_event_type",
    "actor1", "actor2", "assoc_actor_1", "assoc_actor_2",
    "inter1", "inter2", "interaction", "country",
    "admin1", "admin2", "admin3", "location",
    "latitude", "longitude", "geo_precision",
    "source", "source_scale", "notes", "fatalities", "timestamp",
}

### Loading and Schema Validation

`load_raw` reads every `acled_*.csv` from `data/raw/`, inferring the country name from the second underscore-delimited token in the filename (e.g. `acled_myanmar_...` → `Myanmar`).

`check_schema` compares column sets across all four files and flags any asymmetries — a mismatch would mean one country file has extra or missing columns.

In [ ]:
def load_raw() -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    for path in sorted(RAW_DIR.glob("acled_*.csv")):
        country = path.stem.split("_")[1].capitalize()
        df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
        frames[country] = df
        print(f"  {path.name}: {len(df):,} rows x {df.shape[1]} cols")
    return frames


def check_schema(frames: dict[str, pd.DataFrame]) -> list[str]:
    col_sets = {c: set(df.columns) for c, df in frames.items()}
    all_cols: set[str] = set.union(*col_sets.values())
    issues: list[str] = []
    for country, cols in col_sets.items():
        missing = all_cols - cols
        if missing:
            issues.append(f"{country}: missing columns {sorted(missing)}")
    return issues

### Column Profiling and Geo-Precision Diagnostics

`build_col_profile` computes, for each column in the *combined* raw data: % missing, number of unique values, and a random sample of four valid values.

The geo-precision analysis (`geo_retention`, `geo_recommendation`) is the critical output of this step. ACLED codes locations at three precision levels: 1 = exact point, 2 = named area or village approximation, 3 = regional proxy. The 50 % heuristic (`GEO1_FLOOR`) decides which threshold to recommend for the 5 km buffer analysis in Step 2 — if any country falls below 50 % retention at precision 1, the recommendation expands to ≤ 2.

In [ ]:
def pct_missing(s: pd.Series) -> float:
    n = len(s)
    if n == 0:
        return 0.0
    n_null = int(s.isna().sum())
    n_empty = int((s == "").sum()) if s.dtype == object else 0
    return round(100.0 * (n_null + n_empty) / n, 1)


def sample_values(s: pd.Series, k: int = 4) -> list:
    valid = s.dropna()
    if s.dtype == object:
        valid = valid[valid != ""]
    if len(valid) == 0:
        return []
    return valid.sample(min(k, len(valid)), random_state=20260428).tolist()


def build_col_profile(df: pd.DataFrame) -> list[dict]:
    return [
        {
            "column": col,
            "dtype": str(df[col].dtype),
            "pct_missing": pct_missing(df[col]),
            "n_unique": df[col].nunique(dropna=True),
            "examples": sample_values(df[col]),
        }
        for col in df.columns
    ]


def geo_retention(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    for country, df in frames.items():
        n = len(df)
        n1 = int((df["geo_precision"] == 1).sum())
        n2 = int((df["geo_precision"] <= 2).sum())
        rows.append({
            "country": country,
            "n_total": n,
            "n_geo1": n1,
            "pct_geo1": round(100.0 * n1 / n, 1) if n else 0.0,
            "n_geo2": n2,
            "pct_geo2": round(100.0 * n2 / n, 1) if n else 0.0,
        })
    return pd.DataFrame(rows)


def geo_recommendation(ret: pd.DataFrame) -> tuple[str, str]:
    worst = ret.loc[ret["pct_geo1"].idxmin()]
    if worst["pct_geo1"] < GEO1_FLOOR:
        return (
            "<= 2",
            (
                f"geo_precision == 1 retains only **{worst['pct_geo1']:.1f}%** of events "
                f"in {worst['country']}, below the 50% heuristic threshold. "
                f"Precision-2 events are coded to a named area or village approximation "
                f"(ACLED codebook: 'general area or near a town'), which is acceptable "
                f"for 5 km buffer analysis. geo_precision == 3 (regional-level proxies) "
                f"remains excluded."
            ),
        )
    return (
        "== 1",
        (
            f"All countries retain >= 50% of events at geo_precision == 1 "
            f"(minimum: **{worst['pct_geo1']:.1f}%** in {worst['country']}). "
            f"Restricting to the highest-precision locations maximises spatial accuracy "
            f"for the 5 km buffer analysis in Step 2."
        ),
    )


def event_type_dist(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    grp = (
        df.groupby(["event_type", "sub_event_type"], sort=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )
    grp["pct"] = (100.0 * grp["count"] / n).round(1)
    return grp


def fatality_stats(s: pd.Series) -> dict:
    f = pd.to_numeric(s, errors="coerce").dropna()
    q1, q3 = float(f.quantile(0.25)), float(f.quantile(0.75))
    threshold = q3 + 3.0 * (q3 - q1)
    return {
        "min": int(f.min()),
        "max": int(f.max()),
        "mean": round(float(f.mean()), 2),
        "median": float(f.median()),
        "pct_zero": round(100.0 * float((f == 0).mean()), 1),
        "n_outliers": int((f > threshold).sum()),
        "outlier_threshold": round(threshold, 1),
    }

### Cleaning Pipeline and Profile Writer

`clean_and_combine` applies the three cleaning steps to each country independently, records a drop log, and concatenates the results. The Parquet output uses PyArrow for fast columnar reads in subsequent notebooks.

`write_profile` serialises all diagnostic results to a single Markdown report. This report documents the data quality decisions made here so they can be audited without re-running the notebook.

In [ ]:
def clean_and_combine(
    frames: dict[str, pd.DataFrame],
) -> tuple[pd.DataFrame, list[dict]]:
    cleaned: list[pd.DataFrame] = []
    log: list[dict] = []
    for country, raw in frames.items():
        df = raw.copy()
        n_raw = len(df)

        df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
        n_bad_date = int(df["event_date"].isna().sum())
        df = df.dropna(subset=["event_date"])

        n_pre_dedup = len(df)
        df = df.drop_duplicates(subset=["event_id_cnty"], keep="first")
        n_dupes = n_pre_dedup - len(df)

        df["country"] = df["country"].str.strip()

        log.append({
            "country": country,
            "n_raw": n_raw,
            "n_dropped_bad_date": n_bad_date,
            "n_dropped_duplicate": n_dupes,
            "n_clean": len(df),
        })
        cleaned.append(df)

    return pd.concat(cleaned, ignore_index=True), log


def write_profile(
    frames, col_profile, schema_issues, retention,
    threshold, rationale, et_dists, fat_stats_map, drop_log, path,
) -> None:
    lines: list[str] = []
    a = lines.append

    a("# ACLED Data Profile")
    a("")
    a("**Generated:** 2026-04-28  ")
    a("**Source:** `data/raw/` (4 CSV files, downloaded 2026-04-22)  ")
    a("**Countries:** Ecuador, Myanmar, Nigeria, Somalia")
    a("")
    a("---")
    a("")
    a("## 1. File Inventory")
    a("")
    a("| File | Country | Rows | Columns |")
    a("|------|---------|-----:|--------:|")
    for country, df in frames.items():
        a(f"| `acled_{country.lower()}_2026-04-22.csv` | {country} | {len(df):,} | {df.shape[1]} |")
    a("")
    a("### Schema Consistency")
    a("")
    if schema_issues:
        for issue in schema_issues:
            a(f"- **MISMATCH:** {issue}")
    else:
        a("All 4 files have identical schemas. No mismatches found. ✓")
    a("")
    first_df = next(iter(frames.values()))
    extra_cols = [c for c in first_df.columns if c not in STANDARD_COLS]
    if extra_cols:
        a("**Columns present in data but not in the standard-column specification:** "
          + ", ".join(f"`{c}`" for c in extra_cols))
        a("")
        a("These are valid ACLED fields documented in the codebook and retained as-is.")
    a("")
    a("---")
    a("")
    a("## 2. Column Profiles")
    a("")
    a("Profiled on the combined raw data (all 4 countries, before cleaning).")
    a("")
    a("| Column | dtype | % Missing | N Unique | Example Values |")
    a("|--------|-------|----------:|---------:|----------------|")
    for p in col_profile:
        ex = " | ".join(str(e)[:30] for e in p["examples"][:3])
        a(f"| `{p['column']}` | {p['dtype']} | {p['pct_missing']}% "
          f"| {p['n_unique']:,} | {ex} |")
    a("")
    a("---")
    a("")
    a("## 3. Targeted Diagnostics")
    a("")
    a("### 3.1 Latitude/Longitude Missingness")
    a("")
    a("| Country | N Total | Null Lat (n) | Null Lat (%) | Null Lon (n) | Null Lon (%) |")
    a("|---------|--------:|-------------:|-------------:|-------------:|-------------:|")
    for country, df in frames.items():
        n = len(df)
        nl = int(df["latitude"].isna().sum())
        nlo = int(df["longitude"].isna().sum())
        a(f"| {country} | {n:,} | {nl:,} | {100*nl/n:.1f}% | {nlo:,} | {100*nlo/n:.1f}% |")
    a("")
    a("### 3.2 Geo-Precision Distribution by Country")
    a("")
    a("| Country | geo=1 (n) | geo=1 (%) | geo=2 (n) | geo=2 (%) | geo=3 (n) | geo=3 (%) |")
    a("|---------|----------:|----------:|----------:|----------:|----------:|----------:|")
    for country, df in frames.items():
        n = len(df)
        g1 = int((df["geo_precision"] == 1).sum())
        g2 = int((df["geo_precision"] == 2).sum())
        g3 = int((df["geo_precision"] == 3).sum())
        a(f"| {country} | {g1:,} | {100*g1/n:.1f}% "
          f"| {g2:,} | {100*g2/n:.1f}% "
          f"| {g3:,} | {100*g3/n:.1f}% |")
    a("")
    a("### 3.3 Geo-Precision Retention (cumulative)")
    a("")
    a("| Country | geo==1 (n) | geo==1 (%) | geo<=2 (n) | geo<=2 (%) | All events |")
    a("|---------|----------:|----------:|----------:|----------:|-----------:|")
    for _, row in retention.iterrows():
        a(f"| {row['country']} | {row['n_geo1']:,} | {row['pct_geo1']}% "
          f"| {row['n_geo2']:,} | {row['pct_geo2']}% | {row['n_total']:,} |")
    a("")
    a("### 3.4 Event Type and Sub-Event Type Distribution")
    a("")
    for country, dist in et_dists.items():
        a(f"#### {country}")
        a("")
        a("| Event Type | Sub-Event Type | N | % |")
        a("|-----------|---------------|-:|--:|")
        for _, row in dist.iterrows():
            a(f"| {row['event_type']} | {row['sub_event_type']} "
              f"| {row['count']:,} | {row['pct']}% |")
        a("")
    a("### 3.5 Event Date Range and Granularity")
    a("")
    a("| Country | Earliest | Latest | Span (days) |")
    a("|---------|----------|-------:|------------:|")
    for country, df in frames.items():
        dates = pd.to_datetime(df["event_date"], errors="coerce").dropna()
        a(f"| {country} | {dates.min().date()} | {dates.max().date()} "
          f"| {(dates.max() - dates.min()).days:,} |")
    a("")
    a("All files record one event per row with daily precision.")
    a("")
    a("### 3.6 Unique actor1 Values")
    a("")
    a("| Country | N Unique actor1 |")
    a("|---------|----------------:|")
    for country, df in frames.items():
        a(f"| {country} | {df['actor1'].nunique():,} |")
    a("")
    a("### 3.7 Fatalities Distribution")
    a("")
    a("| Country | Min | Max | Mean | Median | % Zero | N Outliers (>Q3+3xIQR) | Outlier Threshold |")
    a("|---------|----:|----:|-----:|-------:|-------:|---------------------:|------------------:|")
    for country, s in fat_stats_map.items():
        a(f"| {country} | {s['min']} | {s['max']} | {s['mean']} | {s['median']} "
          f"| {s['pct_zero']}% | {s['n_outliers']} | {s['outlier_threshold']} |")
    a("")
    a("---")
    a("")
    a("## 4. Cleaning Summary")
    a("")
    a("| Country | Raw rows | Dropped (bad date) | Dropped (dup. ID) | Clean rows |")
    a("|---------|--------:|-----------------:|----------------:|-----------:|")
    for d in drop_log:
        a(f"| {d['country']} | {d['n_raw']:,} | {d['n_dropped_bad_date']:,} "
          f"| {d['n_dropped_duplicate']:,} | {d['n_clean']:,} |")
    a("")
    a("---")
    a("")
    a("## 5. Geo-Precision Recommendation")
    a("")
    a(f"**Recommended threshold: `geo_precision {threshold}`**")
    a("")
    a(rationale)
    a("")
    a("| Country | geo==1 (%) | geo<=2 (%) | All (%) |")
    a("|---------|----------:|----------:|--------:|")
    for _, row in retention.iterrows():
        a(f"| {row['country']} | {row['pct_geo1']}% | {row['pct_geo2']}% | 100.0% |")
    a("")
    a("> **Note:** This threshold is applied in Step 2 as `GEO_THRESH` for all spatial filtering.")

    path.write_text("\n".join(lines), encoding="utf-8")

### Run — Step 1

Loads raw files, runs diagnostics, cleans and saves the combined Parquet, and writes the data profile report. The printed summary shows the geo-precision recommendation — this feeds directly into `GEO_THRESH = 2` in Step 2.

In [ ]:
def main_01() -> None:
    print("=== Step 1: ACLED Data Exploration & Cleaning ===\n")

    print("[1/6] Loading raw files...")
    frames = load_raw()

    print("\n[2/6] Checking schema consistency...")
    schema_issues = check_schema(frames)
    if schema_issues:
        for issue in schema_issues:
            print(f"  WARNING: {issue}")
    else:
        print("  All schemas identical. No mismatches.")

    print("\n[3/6] Building column profile...")
    combined_raw = pd.concat(list(frames.values()), ignore_index=True)
    col_profile = build_col_profile(combined_raw)
    print(f"  Profiled {len(col_profile)} columns across {len(combined_raw):,} total rows")

    print("\n[4/6] Running targeted diagnostics...")
    retention = geo_retention(frames)
    threshold, rationale = geo_recommendation(retention)
    et_dists = {c: event_type_dist(df) for c, df in frames.items()}
    fat_stats_map = {c: fatality_stats(df["fatalities"]) for c, df in frames.items()}
    print("  Diagnostics complete")

    print("\n[5/6] Cleaning and combining...")
    combined, drop_log = clean_and_combine(frames)
    parquet_path = PROCESSED_DIR / "acled_clean.parquet"
    combined.to_parquet(parquet_path, engine="pyarrow", index=False)
    print(f"  Saved acled_clean.parquet: {len(combined):,} rows")

    print("\n[6/6] Writing data profile...")
    write_profile(
        frames=frames, col_profile=col_profile, schema_issues=schema_issues,
        retention=retention, threshold=threshold, rationale=rationale,
        et_dists=et_dists, fat_stats_map=fat_stats_map, drop_log=drop_log,
        path=PROCESSED_DIR / "00_data_profile.md",
    )
    print("  Saved 00_data_profile.md")

    print("\n=== COMPLETE ===")
    print(f"  Geo-precision recommendation: {threshold}")
    for _, row in retention.iterrows():
        print(f"    {row['country']:10s}: geo==1 {row['pct_geo1']:5.1f}%,  "
              f"geo<=2 {row['pct_geo2']:5.1f}%")


main_01()

---

## Step 2 — First-Actor Temporal and Spatial Analysis

**Input:** `data/processed/acled_clean.parquet`

**Output:**
- `output/01_first_actor_analysis.md` — full narrative report per country.
- `output/01_first_actor_summary.csv` — one-row-per-country summary table.
- `output/figures/timeline_<country>.png` — weekly event count + fatalities chart.
- `output/figures/map_<country>.png` — event points and 5 km buffer clusters.

The analysis focuses on `actor1` (initiating actor). For each country, the actor with the most total events is selected. Their activity is then characterised on two dimensions:

**Temporal** — weekly event counts from first-event week to 2026-04-17 are tested for trend (Kendall τ, α = 0.05). A spike-and-vanish pattern is detected first: if ≥ 80 % of events fall within any 12-week window *and* the last 26 weeks are empty, the actor is labelled Spike-and-vanish regardless of the trend test. Remaining actors are classified as Escalating, Declining, Sustained (≥ 60 % active weeks), Sporadic (< 30 %), or Other.

**Spatial** — events at `geo_precision ≤ 2` are projected to the country's UTM zone, buffered 5 km, and dissolved into contiguous clusters. The Largest Cluster Share (LCS = events in the biggest cluster / total events) indicates geographic concentration: ≥ 70 % = Concentrated, < 30 % = Dispersed.

In [ ]:
GEO_THRESH = 2

ATTACK_TYPES = frozenset({
    "Battles",
    "Explosions/Remote violence",
    "Violence against civilians",
})

COUNTRY_CRS: dict[str, str] = {
    "Ecuador": "EPSG:32717",
    "Myanmar": "EPSG:32647",
    "Nigeria": "EPSG:32633",
    "Somalia": "EPSG:32638",
}

DATASET_END = pd.Timestamp("2026-04-17")

SPIKE_WINDOW    = 12
SPIKE_FRAC      = 0.80
VANISH_RECENT   = 26
SUSTAINED_FRAC  = 0.60
SPORADIC_FRAC   = 0.30
MIN_TREND_WEEKS = 10
ALPHA           = 0.05

BUFFER_M         = 5_000
LCS_CONCENTRATED = 70.0
LCS_DISPERSED    = 30.0

### Result Containers

Two named tuples hold the outputs of the classification functions so they can be passed cleanly to the report writer without a loose dictionary.

In [ ]:
class TemporalResult(NamedTuple):
    pattern: str
    tau: float | None
    p_value: float | None
    slope_per_week: float | None
    coverage_pct: float
    n_active_weeks: int
    n_period_weeks: int
    spike_frac: float


class SpatialResult(NamedTuple):
    n_events_geo: int
    n_clusters: int
    buffered_area_km2: float
    lcs_pct: float
    concentration: str
    largest_centroid_lat: float | None
    largest_centroid_lon: float | None

### Temporal Classification

`build_weekly_series` aligns event counts to ISO Monday weeks and forward-fills zeros to the last dataset date, so all actors share a common time axis.

`classify_temporal` applies the decision tree described above. The spike check runs first because a volatile actor can produce a statistically significant trend purely from the spike — labelling it Escalating would be misleading. Kendall τ is used instead of OLS slope because it is robust to the heavily zero-inflated weekly counts typical of conflict data.

In [ ]:
def build_weekly_series(df_actor: pd.DataFrame) -> pd.DataFrame:
    df = df_actor.copy()
    df["iso_monday"] = (
        df["event_date"] - pd.to_timedelta(df["event_date"].dt.dayofweek, unit="D")
    ).dt.normalize()

    weekly = (
        df.groupby("iso_monday")
        .agg(event_count=("event_id_cnty", "count"), fatalities=("fatalities", "sum"))
        .reset_index()
    )

    dataset_end_monday = DATASET_END - pd.Timedelta(days=DATASET_END.weekday())
    all_mondays = pd.date_range(
        start=weekly["iso_monday"].min(),
        end=dataset_end_monday,
        freq="7D",
    )
    full = pd.DataFrame({"iso_monday": all_mondays}).merge(weekly, on="iso_monday", how="left")
    full["event_count"] = full["event_count"].fillna(0).astype(int)
    full["fatalities"]  = full["fatalities"].fillna(0).astype(int)
    return full.reset_index(drop=True)


def classify_temporal(weekly: pd.DataFrame) -> TemporalResult:
    counts   = weekly["event_count"].values.astype(float)
    n        = len(counts)
    total    = counts.sum()
    n_active = int((counts > 0).sum())
    coverage = n_active / n if n > 0 else 0.0

    spike_frac = 0.0
    spike = False
    if total > 0 and n >= SPIKE_WINDOW:
        roll_max = pd.Series(counts).rolling(SPIKE_WINDOW, min_periods=SPIKE_WINDOW).sum().max()
        if not np.isnan(roll_max):
            spike_frac = roll_max / total
            spike = spike_frac >= SPIKE_FRAC
    recent = counts[-VANISH_RECENT:] if n >= VANISH_RECENT else counts
    vanish = recent.sum() == 0

    if spike and vanish:
        return TemporalResult(
            pattern="Spike-and-vanish",
            tau=None, p_value=None, slope_per_week=None,
            coverage_pct=round(100.0 * coverage, 1),
            n_active_weeks=n_active, n_period_weeks=n,
            spike_frac=round(spike_frac, 3),
        )

    tau = p_value = slope_per_week = None
    if n_active >= MIN_TREND_WEEKS:
        tau_val, p_val = stats.kendalltau(np.arange(n), counts)
        slope = float(np.polyfit(np.arange(n), counts, 1)[0])
        tau, p_value, slope_per_week = float(tau_val), float(p_val), round(slope, 4)

    escalating = tau is not None and tau > 0 and p_value < ALPHA
    declining  = tau is not None and tau < 0 and p_value < ALPHA

    if escalating:
        pattern = "Escalating"
    elif declining:
        pattern = "Declining"
    elif coverage >= SUSTAINED_FRAC:
        pattern = "Sustained"
    elif coverage < SPORADIC_FRAC:
        pattern = "Sporadic"
    else:
        pattern = "Other"

    return TemporalResult(
        pattern=pattern,
        tau=round(tau, 4) if tau is not None else None,
        p_value=round(p_value, 4) if p_value is not None else None,
        slope_per_week=slope_per_week,
        coverage_pct=round(100.0 * coverage, 1),
        n_active_weeks=n_active, n_period_weeks=n,
        spike_frac=round(spike_frac, 3),
    )

### Spatial Analysis

Events passing the geo-precision filter are projected to the country's UTM zone (metres), buffered 5 km, and dissolved with `unary_union` into contiguous polygons. Each polygon is a "cluster" — a zone where the actor operated within 5 km of itself at least once.

The **Largest Cluster Share (LCS)** captures whether the actor operates from one base area or across many dispersed locations. A sjoin assigns each point to a cluster; a fallback `sjoin_nearest` handles floating-point edge cases where a buffered point doesn't quite touch the dissolved polygon.

In [ ]:
def compute_spatial_data(
    df_actor: pd.DataFrame, country: str
) -> tuple[SpatialResult, gpd.GeoDataFrame | None, gpd.GeoDataFrame | None]:
    crs = COUNTRY_CRS[country]
    df = df_actor[
        (df_actor["geo_precision"] <= GEO_THRESH)
        & df_actor["latitude"].notna()
        & df_actor["longitude"].notna()
    ].copy()

    n_geo = len(df)
    if n_geo == 0:
        return SpatialResult(0, 0, 0.0, 0.0, "N/A", None, None), None, None

    points = gpd.GeoDataFrame(
        df.reset_index(drop=True),
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    ).to_crs(crs)

    dissolved     = unary_union(list(points.geometry.buffer(BUFFER_M)))
    geoms         = list(dissolved.geoms) if dissolved.geom_type == "MultiPolygon" else [dissolved]
    n_clusters    = len(geoms)
    buffered_area = round(dissolved.area / 1_000_000, 2)

    clusters = gpd.GeoDataFrame({"cluster_id": range(n_clusters)}, geometry=geoms, crs=crs)

    joined    = gpd.sjoin(points[["geometry"]], clusters, how="left", predicate="within")
    unmatched = joined["cluster_id"].isna()
    if unmatched.any():
        nearest = gpd.sjoin_nearest(points[unmatched][["geometry"]], clusters, how="left")
        joined.loc[unmatched, "cluster_id"] = nearest["cluster_id"].values

    cluster_counts = joined["cluster_id"].value_counts()
    largest_id     = int(cluster_counts.idxmax())
    lcs_pct        = round(100.0 * int(cluster_counts.iloc[0]) / n_geo, 1)

    concentration = (
        "Concentrated" if lcs_pct >= LCS_CONCENTRATED
        else "Dispersed" if lcs_pct < LCS_DISPERSED
        else "Mixed"
    )

    centroid_proj = clusters.loc[largest_id, "geometry"].centroid
    centroid_4326 = gpd.GeoDataFrame({"geometry": [centroid_proj]}, crs=crs).to_crs("EPSG:4326")
    centroid_lon  = round(float(centroid_4326.geometry.x.iloc[0]), 4)
    centroid_lat  = round(float(centroid_4326.geometry.y.iloc[0]), 4)

    return (
        SpatialResult(n_geo, n_clusters, buffered_area, lcs_pct, concentration,
                      centroid_lat, centroid_lon),
        points,
        clusters,
    )


def top_actors(df_country: pd.DataFrame, n: int = 3) -> pd.DataFrame:
    grp = (
        df_country.groupby("actor1")
        .agg(
            event_count=("event_id_cnty", "count"),
            total_fatalities=("fatalities", "sum"),
            active_from=("event_date", "min"),
            active_to=("event_date", "max"),
        )
        .sort_values("event_count", ascending=False)
        .head(n)
        .reset_index()
    )
    grp["active_from"] = grp["active_from"].dt.date
    grp["active_to"]   = grp["active_to"].dt.date
    return grp


def _xaxis_locator(n_weeks: int) -> tuple:
    if n_weeks > 520:
        return mdates.YearLocator(2), mdates.DateFormatter("%Y")
    if n_weeks > 260:
        return mdates.YearLocator(), mdates.DateFormatter("%Y")
    if n_weeks > 104:
        return mdates.MonthLocator(bymonth=[1, 7]), mdates.DateFormatter("%b %Y")
    return mdates.MonthLocator(bymonth=range(1, 13, 3)), mdates.DateFormatter("%b %Y")

### Figures

Each first actor gets two figures:

1. **Timeline** — dual-axis bar/line chart of weekly event count (left, blue) and weekly fatalities (right, red). Tick density adapts to the actor's active period length.
2. **Map** — event scatter on the 5 km buffer clusters, projected to UTM. A CartoDB Positron basemap is added if `contextily` is installed (optional).

In [ ]:
def plot_timeline(weekly, actor, country, pattern, path):
    fig, ax1 = plt.subplots(figsize=(13, 4))
    dates = weekly["iso_monday"]

    ax1.bar(dates, weekly["event_count"], color="#4878CF", alpha=0.75, width=6, label="Events")
    ax1.set_ylabel("Weekly Event Count", color="#4878CF", fontsize=10)
    ax1.tick_params(axis="y", labelcolor="#4878CF")
    ax1.set_xlabel("ISO Week", fontsize=10)

    ax2 = ax1.twinx()
    ax2.plot(dates, weekly["fatalities"], color="#D65F5F", linewidth=1.4, alpha=0.85, label="Fatalities")
    ax2.set_ylabel("Weekly Fatalities", color="#D65F5F", fontsize=10)
    ax2.tick_params(axis="y", labelcolor="#D65F5F")

    locator, formatter = _xaxis_locator(len(weekly))
    ax1.xaxis.set_major_locator(locator)
    ax1.xaxis.set_major_formatter(formatter)
    fig.autofmt_xdate(rotation=30)

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9)

    actor_label = actor[:65] + ("…" if len(actor) > 65 else "")
    ax1.set_title(f"{country}  ·  {actor_label}\nTemporal pattern: {pattern}", fontsize=11)
    plt.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_map(points, clusters, actor, country, path):
    if points is None or len(points) == 0:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.text(0.5, 0.5, "No spatial data available", ha="center", va="center",
                transform=ax.transAxes, fontsize=13)
        fig.savefig(path, dpi=150)
        plt.close(fig)
        return

    fig, ax = plt.subplots(figsize=(9, 9))
    clusters.plot(ax=ax, color="#E08C45", alpha=0.30, edgecolor="#B06010", linewidth=0.6)
    points.plot(ax=ax, color="#2B4C7E", markersize=2.5, alpha=0.45)

    if HAS_CONTEXTILY:
        try:
            ctx.add_basemap(ax, crs=points.crs.to_string(),
                            source=ctx.providers.CartoDB.Positron,
                            zoom="auto", attribution_size=6)
        except Exception:
            pass

    actor_label = actor[:65] + ("…" if len(actor) > 65 else "")
    ax.set_title(f"{country}  ·  {actor_label}\nEvent points and 5 km buffer clusters", fontsize=11)
    legend_handles = [
        mpatches.Patch(facecolor="#E08C45", edgecolor="#B06010", alpha=0.5,
                       label=f"5 km buffer clusters (n={len(clusters)})"),
        mlines.Line2D([], [], marker="o", linestyle="none", color="#2B4C7E",
                      markersize=5, alpha=0.7,
                      label=f"Events geo≤{GEO_THRESH} (n={len(points):,})"),
    ]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=9, framealpha=0.8)
    ax.set_axis_off()
    plt.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

### Output Writers

`write_report` produces a Markdown document with one section per country, including ranked actor table, temporal pattern details, and spatial metrics.

`write_summary_csv` produces a compact one-row-per-country CSV for use in presentations or further analysis.

In [ ]:
def write_report(results: list[dict], path: Path) -> None:
    lines: list[str] = []
    a = lines.append

    a("# First-Actor Temporal-Spatial Analysis")
    a("")
    a("**Generated:** 2026-04-28  ")
    a(f"**Geo-precision threshold:** geo_precision <= {GEO_THRESH}  ")
    a(f"**Buffer radius:** {BUFFER_M // 1000} km  ")
    a(f"**Attack event types:** {', '.join(sorted(ATTACK_TYPES))}  ")
    a(f"**Temporal thresholds:** spike ≥{int(SPIKE_FRAC*100)}% in {SPIKE_WINDOW}w; "
      f"sustained ≥{int(SUSTAINED_FRAC*100)}% active weeks; "
      f"sporadic <{int(SPORADIC_FRAC*100)}%; trend p<{ALPHA}  ")
    a(f"**Spatial concentration:** LCS ≥{LCS_CONCENTRATED:.0f}% = Concentrated; "
      f"<{LCS_DISPERSED:.0f}% = Dispersed")
    a("")
    a("---")
    a("")

    for r in results:
        tm = r["temporal"]
        sp = r["spatial"]
        a(f"## {r['country']}")
        a("")
        a("### Top 3 Actor1 by Total Event Count")
        a("")
        a("| Rank | Actor1 | Events | Fatalities | Active From | Active To |")
        a("|-----|--------|-------:|-----------:|------------|----------|")
        for i, row in r["top3"].iterrows():
            marker = " ← **first actor**" if i == 0 else ""
            if i == 1 and r["top3"].iloc[0]["event_count"] > 0:
                gap = (r["top3"].iloc[0]["event_count"] - row["event_count"]) / r["top3"].iloc[0]["event_count"]
                marker += " *(within 5% of rank 1)*" if gap < 0.05 else ""
            a(f"| {i + 1} | {row['actor1']}{marker} | {row['event_count']:,} "
              f"| {row['total_fatalities']:,} | {row['active_from']} | {row['active_to']} |")
        a("")
        a(f"### First Actor: {r['actor']}")
        a("")
        a(f"**Total events (all types):** {r['total_events']:,}  ")
        a(f"**Total fatalities:** {r['total_fatalities']:,}  ")
        a(f"**Active period:** {r['active_from']} → {r['active_to']}  ")
        a(f"**Attack events:** {r['attack_pct']:.1f}% in Battles / Explosions / VAC")
        a("")
        a("#### Temporal Classification")
        a("")
        a(f"**Pattern: {tm.pattern}**")
        a("")
        a(f"- Period weeks: {tm.n_period_weeks:,}  |  Active weeks: {tm.n_active_weeks:,} ({tm.coverage_pct}%)")
        if tm.tau is not None:
            sig = "significant" if tm.p_value < ALPHA else "not significant"
            a(f"- Kendall τ = {tm.tau} (p = {tm.p_value}, {sig})  |  slope = {tm.slope_per_week} ev/week")
        if tm.spike_frac > 0:
            a(f"- Max 12-week window share: {tm.spike_frac * 100:.1f}%")
        a("")
        a("#### Spatial Profile")
        a("")
        a(f"- Events at geo_precision ≤ {GEO_THRESH}: {sp.n_events_geo:,}")
        a(f"- Clusters (5 km buffer dissolve): {sp.n_clusters:,}  |  "
          f"Buffered area: {sp.buffered_area_km2:,} km²")
        a(f"- LCS: {sp.lcs_pct}%  →  **{sp.concentration}**")
        if sp.largest_centroid_lat is not None:
            a(f"- Largest cluster centroid: {sp.largest_centroid_lat}°N, {sp.largest_centroid_lon}°E")
        a("")
        a("---")
        a("")

    path.write_text("\n".join(lines), encoding="utf-8")


def write_summary_csv(results: list[dict], path: Path) -> None:
    rows = [{
        "country": r["country"],
        "first_actor": r["actor"],
        "total_events": r["total_events"],
        "total_fatalities": r["total_fatalities"],
        "active_from": r["active_from"],
        "active_to": r["active_to"],
        "temporal_pattern": r["temporal"].pattern,
        "n_clusters": r["spatial"].n_clusters,
        "buffered_area_km2": r["spatial"].buffered_area_km2,
        "lcs_pct": r["spatial"].lcs_pct,
        "concentration_label": r["spatial"].concentration,
    } for r in results]
    pd.DataFrame(rows).to_csv(path, index=False)

### Run — Step 2

Loops over the four countries, classifies each first actor, generates the timeline and map figures, and writes the report and summary CSV.

In [ ]:
def main_02() -> None:
    print("=== Step 2: First-Actor Temporal-Spatial Analysis ===\n")

    df = pd.read_parquet(PROCESSED_DIR / "acled_clean.parquet")
    print(f"[1/4] Loaded {len(df):,} events across {df['country'].nunique()} countries")

    results: list[dict] = []
    print("\n[2/4] Analysing first actors per country...")
    for country in sorted(df["country"].unique()):
        print(f"\n  ── {country} ──")
        df_c = df[df["country"] == country].copy()

        t3    = top_actors(df_c, n=3)
        actor = t3.iloc[0]["actor1"]
        print(f"  First actor: {actor}  ({t3.iloc[0]['event_count']:,} events)")

        df_a       = df_c[df_c["actor1"] == actor].copy()
        attack_pct = 100.0 * df_a["event_type"].isin(ATTACK_TYPES).mean()

        weekly = build_weekly_series(df_a)
        tm     = classify_temporal(weekly)
        print(f"  Temporal: {tm.pattern}  ({tm.coverage_pct}%, {tm.n_active_weeks}/{tm.n_period_weeks} weeks)")
        if tm.tau is not None:
            print(f"    Kendall τ={tm.tau} (p={tm.p_value}), slope={tm.slope_per_week} ev/wk")

        sp, points_gdf, clusters_gdf = compute_spatial_data(df_a, country)
        print(f"  Spatial: {sp.n_clusters} clusters, LCS {sp.lcs_pct}% → {sp.concentration}")

        results.append({
            "country": country, "actor": actor, "top3": t3,
            "temporal": tm, "spatial": sp,
            "total_events": len(df_a),
            "total_fatalities": int(df_a["fatalities"].sum()),
            "active_from": df_a["event_date"].min().date(),
            "active_to": df_a["event_date"].max().date(),
            "attack_pct": attack_pct,
            "weekly": weekly, "points_gdf": points_gdf, "clusters_gdf": clusters_gdf,
        })

    print("\n[3/4] Generating figures...")
    for r in results:
        tl_path  = FIGURES_DIR / f"timeline_{r['country'].lower()}.png"
        map_path = FIGURES_DIR / f"map_{r['country'].lower()}.png"
        plot_timeline(r["weekly"], r["actor"], r["country"], r["temporal"].pattern, tl_path)
        plot_map(r["points_gdf"], r["clusters_gdf"], r["actor"], r["country"], map_path)
        print(f"  {tl_path.name}  |  {map_path.name}")

    print("\n[4/4] Writing outputs...")
    write_report(results, OUTPUT_DIR / "01_first_actor_analysis.md")
    write_summary_csv(results, OUTPUT_DIR / "01_first_actor_summary.csv")
    print("  01_first_actor_analysis.md  |  01_first_actor_summary.csv")

    print("\n=== COMPLETE ===\n")
    print(f"{'Country':<12} {'First Actor':<45} {'Pattern':<22} {'Clusters':>8}")
    print("-" * 92)
    for r in results:
        print(f"{r['country']:<12} {r['actor'][:44]:<45} "
              f"{r['temporal'].pattern:<22} {r['spatial'].n_clusters:>8}")


main_02()